In [0]:
%pip install scikit-learn

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import pandas as pd
import numpy as np

# Pull the silver table (daily prices + returns)
df = spark.sql("SELECT * FROM silver_stock_analytics ORDER BY Ticker, Date").toPandas()

print(f"Rows loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")

Rows loaded: 63550
Columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'Sector', 'prev_close', 'daily_return_pct']


In [0]:
# FEATURE ENGINEERING
# Create features that describe "what happened recently" for each stock
# The model uses these patterns to predict tomorrow's direction

features_list = []

for ticker in df["Ticker"].unique():
    stock = df[df["Ticker"] == ticker].sort_values("Date").copy()
    
    # Rolling averages of daily returns (momentum indicators)
    stock["return_3d"] = stock["daily_return_pct"].rolling(3).mean()    # avg return last 3 days
    stock["return_5d"] = stock["daily_return_pct"].rolling(5).mean()    # avg return last 5 days
    stock["return_10d"] = stock["daily_return_pct"].rolling(10).mean()  # avg return last 10 days
    
    # Volatility over recent windows
    stock["vol_5d"] = stock["daily_return_pct"].rolling(5).std()        # how wild last 5 days were
    stock["vol_10d"] = stock["daily_return_pct"].rolling(10).std()      # how wild last 10 days were
    
    # Price relative to recent average (is stock above or below its trend?)
    stock["price_vs_5d"] = stock["Close"] / stock["Close"].rolling(5).mean() - 1
    stock["price_vs_20d"] = stock["Close"] / stock["Close"].rolling(20).mean() - 1
    
    # Volume change (unusual trading activity?)
    stock["vol_change"] = stock["Volume"].pct_change()                  # volume vs yesterday
    stock["vol_vs_20d"] = stock["Volume"] / stock["Volume"].rolling(20).mean() - 1  # volume vs 20-day avg
    
    # TARGET: will the stock go UP tomorrow? (1 = yes, 0 = no)
    stock["next_day_return"] = stock["daily_return_pct"].shift(-1)
    stock["target"] = (stock["next_day_return"] > 0).astype(int)
    
    features_list.append(stock)

full_df = pd.concat(features_list).dropna()

feature_cols = ["daily_return_pct", "return_3d", "return_5d", "return_10d",
                "vol_5d", "vol_10d", "price_vs_5d", "price_vs_20d", 
                "vol_change", "vol_vs_20d"]

print(f"Total samples: {len(full_df)}")
print(f"Target distribution:\n{full_df['target'].value_counts()}")
print(f"\nFeature columns: {feature_cols}")
print(f"\nSample data:")
print(full_df[["Date", "Ticker"] + feature_cols + ["target"]].head())

Total samples: 62550
Target distribution:
target
1    32434
0    30116
Name: count, dtype: int64

Feature columns: ['daily_return_pct', 'return_3d', 'return_5d', 'return_10d', 'vol_5d', 'vol_10d', 'price_vs_5d', 'price_vs_20d', 'vol_change', 'vol_vs_20d']

Sample data:
         Date Ticker  daily_return_pct  ...  vol_change  vol_vs_20d  target
19 2021-06-29   AAPL            1.1500  ...    0.039362   -0.125073       1
20 2021-06-30   AAPL            0.4621  ...   -0.020055   -0.144928       1
21 2021-07-01   AAPL            0.2263  ...   -0.170335   -0.279007       1
22 2021-07-02   AAPL            1.9597  ...    0.502361    0.080458       1
23 2021-07-06   AAPL            1.4718  ...    0.371950    0.445568       1

[5 rows x 13 columns]


In [0]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Split data: use everything before 2026 for training, 2026 for testing
# This mimics real life — you train on history and predict the future
train = full_df[full_df["Date"] < "2026-01-01"]
test = full_df[full_df["Date"] >= "2026-01-01"]

X_train = train[feature_cols]
y_train = train["target"]
X_test = test[feature_cols]
y_test = test["target"]

print(f"Training samples: {len(X_train)} (before 2026)")
print(f"Testing samples: {len(X_test)} (2026 onwards)")
print()

# Model 1: Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("=" * 50)
print("MODEL 1: RANDOM FOREST")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, rf_pred):.4f}")
print(f"Precision: {precision_score(y_test, rf_pred):.4f}")
print(f"Recall:    {recall_score(y_test, rf_pred):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, rf_prob):.4f}")
print()

# Model 2: Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_prob = gb.predict_proba(X_test)[:, 1]

print("=" * 50)
print("MODEL 2: GRADIENT BOOSTING")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, gb_pred):.4f}")
print(f"Precision: {precision_score(y_test, gb_pred):.4f}")
print(f"Recall:    {recall_score(y_test, gb_pred):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, gb_prob):.4f}")
print()

# Feature importance (which features matter most?)
importance = pd.DataFrame({
    "Feature": feature_cols,
    "RF_Importance": rf.feature_importances_,
    "GB_Importance": gb.feature_importances_
}).sort_values("GB_Importance", ascending=False)

print("=" * 50)
print("FEATURE IMPORTANCE")
print("=" * 50)
print(importance.to_string(index=False))

Training samples: 56650 (before 2026)
Testing samples: 5900 (2026 onwards)

MODEL 1: RANDOM FOREST
Accuracy:  0.4900
Precision: 0.5007
Recall:    0.8150
AUC-ROC:   0.4880

MODEL 2: GRADIENT BOOSTING
Accuracy:  0.4969
Precision: 0.5058
Recall:    0.6906
AUC-ROC:   0.4970

FEATURE IMPORTANCE
         Feature  RF_Importance  GB_Importance
      vol_vs_20d       0.107251       0.116133
       return_5d       0.102993       0.111425
          vol_5d       0.103390       0.108556
      vol_change       0.106467       0.107658
daily_return_pct       0.105579       0.105117
         vol_10d       0.094292       0.101431
       return_3d       0.092341       0.101018
    price_vs_20d       0.098660       0.094572
      return_10d       0.101098       0.083059
     price_vs_5d       0.087927       0.071030


In [0]:
# IMPROVED FEATURE ENGINEERING


features_list_v2 = []

for ticker in df["Ticker"].unique():
    stock = df[df["Ticker"] == ticker].sort_values("Date").copy()
    
    # Original features
    stock["return_3d"] = stock["daily_return_pct"].rolling(3).mean()
    stock["return_5d"] = stock["daily_return_pct"].rolling(5).mean()
    stock["return_10d"] = stock["daily_return_pct"].rolling(10).mean()
    stock["vol_5d"] = stock["daily_return_pct"].rolling(5).std()
    stock["vol_10d"] = stock["daily_return_pct"].rolling(10).std()
    stock["price_vs_5d"] = stock["Close"] / stock["Close"].rolling(5).mean() - 1
    stock["price_vs_20d"] = stock["Close"] / stock["Close"].rolling(20).mean() - 1
    stock["vol_change"] = stock["Volume"].pct_change()
    stock["vol_vs_20d"] = stock["Volume"] / stock["Volume"].rolling(20).mean() - 1
    
    # NEW: RSI (Relative Strength Index) - classic trading indicator
    delta = stock["Close"].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    stock["rsi_14"] = 100 - (100 / (1 + rs))
    
    # NEW: MACD (Moving Average Convergence Divergence)
    ema_12 = stock["Close"].ewm(span=12).mean()
    ema_26 = stock["Close"].ewm(span=26).mean()
    stock["macd"] = ema_12 - ema_26
    stock["macd_signal"] = stock["macd"].ewm(span=9).mean()
    stock["macd_diff"] = stock["macd"] - stock["macd_signal"]
    
    # NEW: Bollinger Band position (where is price relative to bands?)
    bb_mid = stock["Close"].rolling(20).mean()
    bb_std = stock["Close"].rolling(20).std()
    stock["bb_position"] = (stock["Close"] - bb_mid) / (2 * bb_std)
    
    # NEW: Day of week (markets behave differently on Mondays vs Fridays)
    stock["day_of_week"] = pd.to_datetime(stock["Date"]).dt.dayofweek
    
    # NEW: Month (seasonal patterns)
    stock["month"] = pd.to_datetime(stock["Date"]).dt.month
    
    # NEW: Consecutive up/down days (streak indicator)
    stock["up_streak"] = stock["daily_return_pct"].apply(lambda x: 1 if x > 0 else -1)
    stock["streak"] = stock["up_streak"].groupby(
        (stock["up_streak"] != stock["up_streak"].shift()).cumsum()
    ).cumcount() + 1
    stock["streak"] = stock["streak"] * stock["up_streak"]
    
    # NEW: High-Low range (daily price spread as % of close)
    stock["daily_range"] = (stock["High"] - stock["Low"]) / stock["Close"] * 100
    
    # Target
    stock["next_day_return"] = stock["daily_return_pct"].shift(-1)
    stock["target"] = (stock["next_day_return"] > 0).astype(int)
    
    features_list_v2.append(stock)

full_df_v2 = pd.concat(features_list_v2).dropna()

feature_cols_v2 = ["daily_return_pct", "return_3d", "return_5d", "return_10d",
                   "vol_5d", "vol_10d", "price_vs_5d", "price_vs_20d",
                   "vol_change", "vol_vs_20d",
                   "rsi_14", "macd", "macd_signal", "macd_diff",
                   "bb_position", "day_of_week", "month", "streak", "daily_range"]

# Time-based split
train_v2 = full_df_v2[full_df_v2["Date"] < "2026-01-01"]
test_v2 = full_df_v2[full_df_v2["Date"] >= "2026-01-01"]

X_train_v2 = train_v2[feature_cols_v2]
y_train_v2 = train_v2["target"]
X_test_v2 = test_v2[feature_cols_v2]
y_test_v2 = test_v2["target"]

# Train improved models
rf_v2 = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=50, random_state=42)
rf_v2.fit(X_train_v2, y_train_v2)
rf_pred_v2 = rf_v2.predict(X_test_v2)
rf_prob_v2 = rf_v2.predict_proba(X_test_v2)[:, 1]

gb_v2 = GradientBoostingClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, 
                                     min_samples_leaf=50, random_state=42)
gb_v2.fit(X_train_v2, y_train_v2)
gb_pred_v2 = gb_v2.predict(X_test_v2)
gb_prob_v2 = gb_v2.predict_proba(X_test_v2)[:, 1]

print("=" * 50)
print("IMPROVED MODEL 1: RANDOM FOREST V2")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test_v2, rf_pred_v2):.4f}")
print(f"Precision: {precision_score(y_test_v2, rf_pred_v2):.4f}")
print(f"Recall:    {recall_score(y_test_v2, rf_pred_v2):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test_v2, rf_prob_v2):.4f}")
print()

print("=" * 50)
print("IMPROVED MODEL 2: GRADIENT BOOSTING V2")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test_v2, gb_pred_v2):.4f}")
print(f"Precision: {precision_score(y_test_v2, gb_pred_v2):.4f}")
print(f"Recall:    {recall_score(y_test_v2, gb_pred_v2):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test_v2, gb_prob_v2):.4f}")
print()

# Feature importance for best model
importance_v2 = pd.DataFrame({
    "Feature": feature_cols_v2,
    "GB_Importance": gb_v2.feature_importances_
}).sort_values("GB_Importance", ascending=False)

print("=" * 50)
print("TOP FEATURES (GRADIENT BOOSTING)")
print("=" * 50)
print(importance_v2.to_string(index=False))

IMPROVED MODEL 1: RANDOM FOREST V2
Accuracy:  0.4988
Precision: 0.5063
Recall:    0.7828
AUC-ROC:   0.4936

IMPROVED MODEL 2: GRADIENT BOOSTING V2
Accuracy:  0.4929
Precision: 0.5033
Recall:    0.6107
AUC-ROC:   0.4947

TOP FEATURES (GRADIENT BOOSTING)
         Feature  GB_Importance
       return_5d       0.070482
     daily_range       0.070259
      vol_vs_20d       0.070226
          vol_5d       0.065349
       return_3d       0.063617
daily_return_pct       0.062869
      vol_change       0.062862
           month       0.057517
         vol_10d       0.053443
     day_of_week       0.052792
      return_10d       0.049047
     price_vs_5d       0.048769
       macd_diff       0.046915
            macd       0.045233
     macd_signal       0.042811
     bb_position       0.042766
          rsi_14       0.042680
    price_vs_20d       0.037591
          streak       0.014771


In [0]:
# Save predictions to Databricks for dashboard use
test_results = test_v2[["Date", "Ticker", "Sector", "Close", "daily_return_pct", "target"]].copy()
test_results["rf_prediction"] = rf_pred_v2
test_results["rf_probability"] = rf_prob_v2
test_results["gb_prediction"] = gb_pred_v2
test_results["gb_probability"] = gb_prob_v2

# Save as table
spark_results = spark.createDataFrame(test_results)
spark_results.write.mode("overwrite").saveAsTable("gold_ml_predictions")

print(f"Saved {len(test_results)} predictions to gold_ml_predictions")
print(f"\nSample predictions:")
print(test_results[["Date", "Ticker", "Close", "target", "gb_prediction", "gb_probability"]].head(10).to_string(index=False))


Saved 5900 predictions to gold_ml_predictions

Sample predictions:
      Date Ticker      Close  target  gb_prediction  gb_probability
2026-01-02   AAPL 270.507416       0              0        0.485800
2026-01-05   AAPL 266.764374       0              0        0.480431
2026-01-06   AAPL 261.873444       0              0        0.428379
2026-01-07   AAPL 259.847198       0              0        0.399198
2026-01-08   AAPL 258.559631       1              1        0.636718
2026-01-09   AAPL 258.889008       1              1        0.571973
2026-01-12   AAPL 259.767395       1              0        0.425312
2026-01-13   AAPL 260.565887       0              0        0.472381
2026-01-14   AAPL 259.477905       0              0        0.491347
2026-01-15   AAPL 257.731140       0              1        0.503831
